# Chạy suy luận (Inference) với mô hình đã fine-tune (LoRA)
Sử dụng notebook này để load trọng số LoRA bạn vừa train xong và test thử với các câu hỏi.

In [ ]:
# Nếu bạn chạy ở server mới hoàn toàn, cần cài lại unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir
!pip install transformers trl peft accelerate bitsandbytes xformers --no-cache-dir

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

# ĐƯỜNG DẪN TỚI THƯ MỤC CHỨA TRỌNG SỐ LORA BẠN VỪA TRAIN
# Copy thư mục "qwen2.5-1.5b-instruct-lora-vihealthqa" từ server cũ sang server mới và đặt cùng thư mục với notebook này
lora_path = "qwen2.5-1.5b-instruct-lora-vihealthqa"

print("Đang tải mô hình từ:", lora_path)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = lora_path, # Unsloth sẽ tự động đọc config để tải base model và ghép LoRA vào
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Bật chế độ suy luận nhanh gấp 2 lần của Unsloth
FastLanguageModel.for_inference(model)

In [ ]:
# Hàm tiện ích để hỏi đáp
def ask_doctor_ai(question):
    messages = [
        {"role": "system", "content": "Bạn là một chuyên gia y tế AI thông minh, tận tâm và có khả năng giải đáp các vấn đề sức khỏe một cách chính xác."},
        {"role": "user", "content": question},
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, 
        tokenize=True, 
        add_generation_prompt=True, 
        return_tensors="pt"
    ).to("cuda")
    
    # Generate text
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True)
    
    # Decode kết quả
    input_length = inputs.shape[1]
    answer = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return answer

In [ ]:
# Test thử với các câu hỏi
questions = [
    "Đang chích ngừa viêm gan B có chích ngừa Covid-19 được không?",
    "Bé 13 tháng tuổi uống thuốc Acyclovir có được không?",
    "Tôi hay bị đau đầu, chóng mặt và buồn nôn vào buổi sáng là bệnh gì?"
]

for i, q in enumerate(questions, 1):
    print(f"\n[ĐANG CHẠY CÂU {i}/{len(questions)}]")
    print("\n=======================================")
    print(f"🟢 BỆNH NHÂN HỎI: {q}")
    print(f"🔵 BÁC SĨ AI ĐÁP: \n{ask_doctor_ai(q)}")
    print("=======================================")